In [1]:
import pandas as pd
import numpy as np
import os

folder_path = r"D:\Swapnil\Work\Projects\FIFA\Updated Base Files"

sales = pd.read_csv(os.path.join(folder_path, "Step5_Global_With_FIFA.csv"))

print(sales.shape)
print(sales.columns)

(1389312, 33)
Index(['sales_id', 'date', 'retailer', 'retailer_id', 'us_region', 'state',
       'city', 'product', 'price_per_unit', 'units_sold', 'total_sales',
       'operating_profit', 'operating_margin', 'country_id', 'country',
       'region', 'confederation', 'is_world_cup_team', 'is_host', 'tier',
       'country_multiplier', 'company_name', 'sim_units_sold',
       'sim_price_per_unit', 'sim_total_sales', 'sim_operating_profit',
       'match_day_flag', 'match_count', 'fifa_total_sales', 'fifa_units_sold',
       'fifa_operating_profit', 'knockout_stage_flag',
       'fifa_sales_uplift_pct'],
      dtype='object')


In [2]:
country_summary = sales.groupby(
    ["country", "region", "confederation", "tier", "is_host"],
    as_index=False
).agg({
    "fifa_total_sales": "sum",
    "fifa_operating_profit": "sum",
    "campaign_spend": "sum" if "campaign_spend" in sales.columns else "count",
    "match_day_flag": "mean",
    "fifa_sales_uplift_pct": "mean"
})

KeyError: "Column(s) ['campaign_spend'] do not exist"

In [3]:
country_summary = sales.groupby(
    ["country", "region", "confederation", "tier", "is_host"],
    as_index=False
).agg({
    "fifa_total_sales": "sum",
    "fifa_operating_profit": "sum",
    "match_day_flag": "mean",
    "fifa_sales_uplift_pct": "mean"
})

In [4]:
country_summary = country_summary.rename(columns={
    "fifa_total_sales": "total_revenue",
    "fifa_operating_profit": "total_profit",
    "match_day_flag": "match_day_rate",
    "fifa_sales_uplift_pct": "avg_fifa_uplift_pct"
})

In [5]:
def normalize(series):
    if series.max() == series.min():
        return series * 0
    return (series - series.min()) / (series.max() - series.min())

country_summary["revenue_score"] = normalize(country_summary["total_revenue"])
country_summary["profit_score"] = normalize(country_summary["total_profit"])
country_summary["uplift_score"] = normalize(country_summary["avg_fifa_uplift_pct"])

country_summary["opportunity_score"] = (
    0.45 * country_summary["revenue_score"] +
    0.35 * country_summary["profit_score"] +
    0.20 * country_summary["uplift_score"]
)

In [6]:
country_summary = country_summary.sort_values(
    "opportunity_score",
    ascending=False
).reset_index(drop=True)

country_summary["rank"] = range(1, len(country_summary) + 1)

top10_countries = country_summary.head(10).copy()

In [7]:
def country_recommendation(row):
    if row["is_host"] == 1 and row["opportunity_score"] >= 0.75:
        return "Prioritize host-market campaigns and premium match-day promotions"
    elif row["avg_fifa_uplift_pct"] >= 20:
        return "Increase match-day promotions and localized offers"
    elif row["total_profit"] > country_summary["total_profit"].median():
        return "Maintain strong spend and optimize campaign efficiency"
    else:
        return "Test targeted discounts before scaling investment"

top10_countries["recommendation"] = top10_countries.apply(country_recommendation, axis=1)

In [8]:
company_country_summary = sales.groupby(
    ["company_name", "country", "region", "confederation", "tier", "is_host"],
    as_index=False
).agg({
    "fifa_total_sales": "sum",
    "fifa_operating_profit": "sum",
    "match_day_flag": "mean",
    "fifa_sales_uplift_pct": "mean"
})

company_country_summary = company_country_summary.rename(columns={
    "fifa_total_sales": "total_revenue",
    "fifa_operating_profit": "total_profit",
    "match_day_flag": "match_day_rate",
    "fifa_sales_uplift_pct": "avg_fifa_uplift_pct"
})

company_country_summary["revenue_score"] = normalize(company_country_summary["total_revenue"])
company_country_summary["profit_score"] = normalize(company_country_summary["total_profit"])
company_country_summary["uplift_score"] = normalize(company_country_summary["avg_fifa_uplift_pct"])

company_country_summary["opportunity_score"] = (
    0.45 * company_country_summary["revenue_score"] +
    0.35 * company_country_summary["profit_score"] +
    0.20 * company_country_summary["uplift_score"]
)

company_country_summary = company_country_summary.sort_values(
    "opportunity_score",
    ascending=False
).reset_index(drop=True)

company_country_summary["rank"] = range(1, len(company_country_summary) + 1)

top10_company_country = company_country_summary.head(10).copy()

In [9]:
country_company_matrix = company_country_summary.pivot_table(
    index="country",
    columns="company_name",
    values="total_revenue",
    aggfunc="sum"
).reset_index()

country_company_matrix["top_company"] = country_company_matrix[
    ["Coca-Cola", "Pepsi", "Red Bull"]
].idxmax(axis=1)

In [10]:
country_company_matrix = company_country_summary.pivot_table(
    index="country",
    columns="company_name",
    values="total_revenue",
    aggfunc="sum"
).reset_index()

country_company_matrix["top_company"] = country_company_matrix[
    ["Coca-Cola", "Pepsi", "Red Bull"]
].idxmax(axis=1)

In [11]:
output_path = os.path.join(folder_path, "Step6_Top10_Markets.xlsx")

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    top10_countries.to_excel(writer, sheet_name="Top10_Countries", index=False)
    top10_company_country.to_excel(writer, sheet_name="Top10_Company_Country", index=False)
    country_summary.to_excel(writer, sheet_name="All_Country_Rankings", index=False)
    company_country_summary.to_excel(writer, sheet_name="Company_Country_Rankings", index=False)
    country_company_matrix.to_excel(writer, sheet_name="Country_Company_Matrix", index=False)

print("Saved:", output_path)

Saved: D:\Swapnil\Work\Projects\FIFA\Updated Base Files\Step6_Top10_Markets.xlsx
